In [ ]:
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

%load_ext autoreload
%autoreload 2

# CRITICAL: Preload libcuda.so.1 from system path BEFORE anything else
import ctypes
try:
    ctypes.CDLL("/usr/lib64/libcuda.so.1", mode=ctypes.RTLD_GLOBAL)
    print("✓ Preloaded libcuda.so.1 from /usr/lib64")
except Exception as e:
    print(f"⚠ Could not preload libcuda.so.1: {e}")

# CuPy: ensure CUDA_PATH and LD_LIBRARY_PATH
if "CUDA_PATH" not in os.environ:
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True, executable='/bin/bash', capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            raise RuntimeError("CUDA_PATH not set. Run: module load cuda/12.2")
    except Exception as e:
        raise RuntimeError(f"Failed to load CUDA: {e}") from e

cuda_path = os.environ.get('CUDA_PATH')
if cuda_path:
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),
    ]
    current_ld = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld.split(':') if current_ld else []
    for p in cuda_lib_paths:
        if os.path.exists(p) and p not in ld_paths:
            ld_paths.insert(0, p)
    if ld_paths != (current_ld.split(':') if current_ld else []):
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH")
    for p in cuda_lib_paths:
        if not os.path.exists(p):
            continue
        for name, lib in [('libcudart.so', None), ('libcudart.so.12', None), ('libnvrtc.so.12', None)]:
            path = os.path.join(p, name)
            if os.path.exists(path):
                try:
                    ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
                    print(f"✓ Preloaded {name}")
                except Exception as e:
                    print(f"⚠ Could not preload {name}: {e}")
                break

print("\n=== Environment before CuPy ===")
print(f"CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
print("=====================================\n")

from src.utils.array_backend import np, random, is_cupy
from src.belief_quantized.belief_mdp_n_M import BeliefMDP_n_M_Localization
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from src.belief_quantized.value_iteration import ValueIteration
from tqdm import tqdm
import time
import warnings
import json
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print("✓ All imports successful")

if is_cupy:
    try:
        import cupy as cp
        print("Using backend: CuPy (GPU)")
        print(f"✓ CuPy {cp.__version__}, CUDA devices: {cp.cuda.runtime.getDeviceCount()}")
    except Exception as e:
        print(f"✗ CuPy check failed: {e}")
else:
    print("Using backend: NumPy (CPU)")

"""
Localization value function vs (n, M). Progress is saved after each (n, M) so
a failed or OOM run does not lose prior results. Six cells run the six
quantization levels; you can adjust j_batch_size between cells to control memory.
"""

CWD: /global/home/hpc5656/SLAM
✓ Preloaded libcuda.so.1 from /usr/lib64
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Updated LD_LIBRARY_PATH
✓ Preloaded libcudart.so
✓ Preloaded libcudart.so

=== Environment before CuPy ===
CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2



In [2]:
# Shared configuration (same environment for all n, M)
beta = 0.95
epsilon = 1e-6

# Six quantization levels: (n, M) = (2,2), (2,3), (3,2), (3,3), (4,2), (4,3)
n_list = [2, 3, 4]
M_list = [2, 3]

# Belief index for scalar "optimal cost" (like "initial state x = 1")
initial_belief_index = 0

# Batch size for p_n_M computation: None = process all at once; set to int (e.g. 50) to reduce memory.
# Change this between run cells if you hit OOM on larger (n, M).
j_batch_size = None

obstacles, area = load_obstacles_config(environment='toy2')
motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
sensor = LIDAR(fov=360, r_max=10.0, B=8)

out_dir = PROJECT_ROOT / 'notebooks' / 'localization' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
results_path = out_dir / 'value_sweep_results_LOC.json'

# Load previous progress so a failed run doesn't lose prior results
if results_path.exists():
    with open(results_path, 'r') as f:
        results = json.load(f)
    print(f"Loaded {len(results)} result(s) from {results_path.name}")
else:
    results = []
    print("No previous results; starting fresh.")

print(f"Sweep: n = {n_list}, M = {M_list} (6 levels)")
print(f"β = {beta}, ε = {epsilon}, j_batch_size = {j_batch_size}")
print(f"Environment: toy2, area = {area}")

No previous results; starting fresh.
Sweep: n = [2, 3, 4], M = [2, 3] (6 levels)
β = 0.95, ε = 1e-06, j_batch_size = None
Environment: toy2, area = (0, 10, 0, 10)


In [3]:
# Level 1/6: (n=2, M=2)
n, M = 2, 2
if any(r.get('n') == n and r.get('M') == M for r in results):
    print(f"Already have (n={n}, M={M}); skipping.")
else:
    if j_batch_size is not None:
        BeliefMDP_n_M_Localization._test_j_batch_size = j_batch_size
    try:
        grid_map = LidarGridMapVec(x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3], quantization_level=n)
        bmdp = BeliefMDP_n_M_Localization(M=M, β=beta, n=n, motion_model=motion_model, measurement_model=sensor,
                                          obstacles=obstacles, _map=grid_map, sigma_v=1.0)
        bmdp.set_known_map(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else np.array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(np.mean(V))
        results.append({'n': n, 'M': M, 'm_n': int(bmdp.SQ.m_n), 'cardinality': int(bmdp.BQ.cardinality),
                        'value_initial': v0, 'value_mean': float(np.mean(V)), 'iterations': int(vi.iteration_count)})
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved (n={n}, M={M}); {len(results)} total.")
    except Exception as e:
        print(f"Error (n={n}, M={M}): {e}")
        raise

Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map2x2_max2.0_ea380e8a.npz
  Checking cache file: Q_n_n2_obs2_map2x2_B8_e33751ba.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs2_map2x2_B8_e33751ba.npz
  ✓ Cache validation passed
  ✓ Loaded codebook from cache: cache/belief_quantizer/belief_quantizer_M2_N16.npz
Loaded cached p_n_M from /global/home/hpc5656/SLAM/cache/p_n_M_localization/p_n_M_localization_M2_n2_map2x2_max2.0_6cc95d5e.npz
Computing c_n_M for the first time...
This will compute costs for 136 beliefs × 4 actions
  Computing costs for 136 beliefs × 4 actions...
Saved c_n_M to /global/home/hpc5656/SLAM/cache/c_n_M_localization/c_n_M_localization_M2_n2_map2x2_max2.0_f85fbe04.npz
  Value iteration: problem=localization, |Π|=136, n_u=4, ε=1e-06
    iter 1: max |V - V_old| = 2.02e+00
    iter 2: max |V - V_old| = 1.92e+00
    iter 3: max |V - V_old| = 1.82e+00
    iter 4: max |V - V_old| = 1

In [ ]:
# Optional: run full sweep in parallel on multiple GPUs (one (n,M) per GPU).
# Run the config cell first. Then run this cell to use all 8 GPUs instead of running each level cell by hand.
from src.utils.multi_gpu import run_sweep_parallel_localization

sweep_config = {
    'project_root': str(PROJECT_ROOT),
    'environment': 'toy2',
    'beta': beta,
    'epsilon': epsilon,
    'motion_model_kw': {'p_x': 5.0, 'p_y': 5.0, 'v_x': 0.0, 'v_y': 0.0, 'dt': 1.0, 'max_a': 2.0},
    'sensor_kw': {'fov': 360, 'r_max': 10.0, 'B': 8},
    'initial_belief_index': initial_belief_index,
    'results_path': str(results_path),
    'j_batch_size': j_batch_size,
}
parallel_results = run_sweep_parallel_localization(n_list, M_list, sweep_config, num_gpus=8)
for r in parallel_results:
    if r.get('error'):
        print(f"Error (n={r['n']}, M={r['M']}): {r['error']}")
        continue
    existing = [i for i, x in enumerate(results) if x.get('n') == r['n'] and x.get('M') == r['M']]
    if existing:
        results[existing[0]] = r
    else:
        results.append(r)
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Parallel sweep done; {len(results)} total results saved.")

In [4]:
# Level 2/6: (n=3, M=2)
n, M = 3, 2
if any(r.get('n') == n and r.get('M') == M for r in results):
    print(f"Already have (n={n}, M={M}); skipping.")
else:
    if j_batch_size is not None:
        BeliefMDP_n_M_Localization._test_j_batch_size = j_batch_size
    try:
        grid_map = LidarGridMapVec(x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3], quantization_level=n)
        bmdp = BeliefMDP_n_M_Localization(M=M, β=beta, n=n, motion_model=motion_model, measurement_model=sensor,
                                          obstacles=obstacles, _map=grid_map, sigma_v=1.0)
        bmdp.set_known_map(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else np.array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(np.mean(V))
        results.append({'n': n, 'M': M, 'm_n': int(bmdp.SQ.m_n), 'cardinality': int(bmdp.BQ.cardinality),
                        'value_initial': v0, 'value_mean': float(np.mean(V)), 'iterations': int(vi.iteration_count)})
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved (n={n}, M={M}); {len(results)} total.")
    except Exception as e:
        print(f"Error (n={n}, M={M}): {e}")
        raise

Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n3_map3x3_max2.0_c2818dab.npz
  Cache file does not exist: /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n3_obs3_map3x3_B8_32385ff4.npz
Computing Q_n for the first time...


Computing Q_n:   0%|          | 0/6561 [00:00<?, ?it/s]

Saved Q_n to /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n3_obs3_map3x3_B8_32385ff4.npz
  Generating new codebook: M=2, N_n=81, cardinality=3,321
  Generating codebook: M=2, N_n=81, cardinality=3,321
  Array allocation: 0.000527s
  Iterator creation: 0.000005s
  Starting combination iteration...


Generating combinations: 100%|██████████| 3321/3321 [00:01<00:00, 3293.29it/s]

  Combination iteration: 1.011930s
  Normalization: 0.040601s
  ✓ Codebook generation complete!
  ✓ Saved codebook to cache: cache/belief_quantizer/belief_quantizer_M2_N81.npz
  ✓ Cache file size: 0.0 MB
Computing p_n_M for the first time...
This may take a while: 9 actions × 3321² belief transitions
Computing p_n_M for action 1/9...
  This will compute transitions for 3,321 belief states


...

  Action 7/9: 629 non-zeros (99.99% sparse)
Computing p_n_M for action 8/9...
  This will compute transitions for 3,321 belief states


Action 8/9:   0%|          | 0/3321 [00:00<?, ?it/s]

  Action 8/9: 3,089 non-zeros (99.97% sparse)
Computing p_n_M for action 9/9...
  This will compute transitions for 3,321 belief states


Action 9/9:   0%|          | 0/3321 [00:00<?, ?it/s]

  Action 9/9: 629 non-zeros (99.99% sparse)
Saved p_n_M to /global/home/hpc5656/SLAM/cache/p_n_M_localization/p_n_M_localization_M2_n3_map3x3_max2.0_026893fe.npz
Computing c_n_M for the first time...
This will compute costs for 3321 beliefs × 9 actions
  Computing costs for 3,321 beliefs × 9 actions...
Saved c_n_M to /global/home/hpc5656/SLAM/cache/c_n_M_localization/c_n_M_localization_M2_n3_map3x3_max2.0_26461488.npz
  Value iteration: problem=localization, |Π|=3321, n_u=9, ε=1e-06
    iter 1: max |V - V_old| = 6.93e-01
    iter 2: max |V - V_old| = 6.58e-01
    iter 3: max |V - V_old| = 6.26e-01
    iter 4: max |V - V_old| = 3.79e-01
    iter 5: max |V - V_old| = 7.09e-06
    iter 6: max |V - V_old| = 2.43e-16
  Converged in 6 iterations (max_diff = 2.43e-16)
Saved value iteration results to /global/home/hpc5656/SLAM/cache/LOC/value_iteration/value_iteration_localization_M2_n3_beta0.95_eps1e-06_map3x3_max2.0_ec2a9cb9.npz
  File size: 0.01 MB
  Iterations: 6
  Final max |V - V_old|: 2

In [5]:
# Level 3/6: (n=2, M=3)
n, M = 2, 3
if any(r.get('n') == n and r.get('M') == M for r in results):
    print(f"Already have (n={n}, M={M}); skipping.")
else:
    if j_batch_size is not None:
        BeliefMDP_n_M_Localization._test_j_batch_size = j_batch_size
    try:
        grid_map = LidarGridMapVec(x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3], quantization_level=n)
        bmdp = BeliefMDP_n_M_Localization(M=M, β=beta, n=n, motion_model=motion_model, measurement_model=sensor,
                                          obstacles=obstacles, _map=grid_map, sigma_v=1.0)
        bmdp.set_known_map(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else np.array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(np.mean(V))
        results.append({'n': n, 'M': M, 'm_n': int(bmdp.SQ.m_n), 'cardinality': int(bmdp.BQ.cardinality),
                        'value_initial': v0, 'value_mean': float(np.mean(V)), 'iterations': int(vi.iteration_count)})
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved (n={n}, M={M}); {len(results)} total.")
    except Exception as e:
        print(f"Error (n={n}, M={M}): {e}")
        raise

Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map2x2_max2.0_ea380e8a.npz
  Checking cache file: Q_n_n2_obs2_map2x2_B8_e33751ba.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs2_map2x2_B8_e33751ba.npz
  ✓ Cache validation passed
  ✓ Loaded codebook from cache: cache/belief_quantizer/belief_quantizer_M3_N16.npz
Computing p_n_M for the first time...
This may take a while: 4 actions × 816² belief transitions
Computing p_n_M for action 1/4...
  This will compute transitions for 816 belief states


Action 1/4:   0%|          | 0/816 [00:00<?, ?it/s]

  Action 1/4: 1,744 non-zeros (99.74% sparse)
Computing p_n_M for action 2/4...
  This will compute transitions for 816 belief states


Action 2/4:   0%|          | 0/816 [00:00<?, ?it/s]

  Action 2/4: 1,744 non-zeros (99.74% sparse)
Computing p_n_M for action 3/4...
  This will compute transitions for 816 belief states


Action 3/4:   0%|          | 0/816 [00:00<?, ?it/s]

  Action 3/4: 1,744 non-zeros (99.74% sparse)
Computing p_n_M for action 4/4...
  This will compute transitions for 816 belief states


Action 4/4:   0%|          | 0/816 [00:00<?, ?it/s]

  Action 4/4: 1,744 non-zeros (99.74% sparse)
Saved p_n_M to /global/home/hpc5656/SLAM/cache/p_n_M_localization/p_n_M_localization_M3_n2_map2x2_max2.0_357f82ee.npz
Computing c_n_M for the first time...
This will compute costs for 816 beliefs × 4 actions
  Computing costs for 816 beliefs × 4 actions...
Saved c_n_M to /global/home/hpc5656/SLAM/cache/c_n_M_localization/c_n_M_localization_M3_n2_map2x2_max2.0_23851a90.npz
  Value iteration: problem=localization, |Π|=816, n_u=4, ε=1e-06
    iter 1: max |V - V_old| = 2.42e+00
    iter 2: max |V - V_old| = 2.30e+00
    iter 3: max |V - V_old| = 1.74e+00
    iter 4: max |V - V_old| = 1.63e+00
    iter 5: max |V - V_old| = 1.52e+00
    iter 10: max |V - V_old| = 1.10e+00
    iter 20: max |V - V_old| = 5.97e-01
    iter 30: max |V - V_old| = 3.34e-01
    iter 40: max |V - V_old| = 1.92e-01
    iter 50: max |V - V_old| = 1.12e-01
    iter 60: max |V - V_old| = 6.59e-02
    iter 70: max |V - V_old| = 3.90e-02
    iter 80: max |V - V_old| = 2.32e-02

In [59]:
# Level 4/6: (n=3, M=3)
# To refresh belief_mdp_n_M after editing it: run the "Optional: refresh modules" cell, then re-run this cell.
# To use a different j_batch_size for this run only: set j_batch_size = 50 (or any int) below, then run this cell.
# j_batch_size = 50  # uncomment to override for this run only (no kernel restart)
n, M = 3, 3
if any(r.get('n') == n and r.get('M') == M for r in results):
    print(f"Already have (n={n}, M={M}); skipping.")
else:
    if j_batch_size is not None:
        BeliefMDP_n_M_Localization._test_j_batch_size = j_batch_size
    try:
        grid_map = LidarGridMapVec(x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3], quantization_level=n)
        bmdp = BeliefMDP_n_M_Localization(M=M, β=beta, n=n, motion_model=motion_model, measurement_model=sensor,
                                          obstacles=obstacles, _map=grid_map, sigma_v=1.0)
        bmdp.set_known_map(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else np.array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(np.mean(V))
        results.append({'n': n, 'M': M, 'm_n': int(bmdp.SQ.m_n), 'cardinality': int(bmdp.BQ.cardinality),
                        'value_initial': v0, 'value_mean': float(np.mean(V)), 'iterations': int(vi.iteration_count)})
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved (n={n}, M={M}); {len(results)} total.")
    except Exception as e:
        print(f"Error (n={n}, M={M}): {e}")
        raise

Error (n=3, M=3): cudaErrorIllegalAddress: an illegal memory access was encountered


CUDARuntimeError: cudaErrorIllegalAddress: an illegal memory access was encountered

In [57]:
# Optional: refresh modules without restarting the kernel.
# Run this after editing src/classes/belief_mdp_n_M.py or src/classes/mapping.py to pick up code changes.
# To only change j_batch_size: edit j_batch_size in the config cell above, re-run that cell, then re-run the (n, M) cell.
import importlib
import src.classes.mapping as _mapping_mod
import src.belief_quantized.belief_mdp_n_M as _bmdp_mod
importlib.reload(_mapping_mod)
importlib.reload(_bmdp_mod)

from src.belief_quantized.belief_mdp_n_M import BeliefMDP_n_M_Localization
from src.classes.mapping import LidarGridMapVec
print("Reloaded mapping + BeliefMDP_n_M_Localization")

Reloaded mapping + BeliefMDP_n_M_Localization


In [58]:
# Level 5/6: (n=4, M=2)
n, M = 4, 2
if any(r.get('n') == n and r.get('M') == M for r in results):
    print(f"Already have (n={n}, M={M}); skipping.")
else:
    if j_batch_size is not None:
        BeliefMDP_n_M_Localization._test_j_batch_size = j_batch_size
    try:
        grid_map = LidarGridMapVec(x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3], quantization_level=n)
        bmdp = BeliefMDP_n_M_Localization(M=M, β=beta, n=n, motion_model=motion_model, measurement_model=sensor,
                                          obstacles=obstacles, _map=grid_map, sigma_v=1.0)
        bmdp.set_known_map(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else np.array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(np.mean(V))
        results.append({'n': n, 'M': M, 'm_n': int(bmdp.SQ.m_n), 'cardinality': int(bmdp.BQ.cardinality),
                        'value_initial': v0, 'value_mean': float(np.mean(V)), 'iterations': int(vi.iteration_count)})
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved (n={n}, M={M}); {len(results)} total.")
    except Exception as e:
        print(f"Error (n={n}, M={M}): {e}")
        raise

Error (n=4, M=2): cudaErrorIllegalAddress: an illegal memory access was encountered


CUDARuntimeError: cudaErrorIllegalAddress: an illegal memory access was encountered

In [ ]:
# Level 6/6: (n=4, M=3)
n, M = 4, 3
if any(r.get('n') == n and r.get('M') == M for r in results):
    print(f"Already have (n={n}, M={M}); skipping.")
else:
    if j_batch_size is not None:
        BeliefMDP_n_M_Localization._test_j_batch_size = j_batch_size
    try:
        grid_map = LidarGridMapVec(x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3], quantization_level=n)
        bmdp = BeliefMDP_n_M_Localization(M=M, β=beta, n=n, motion_model=motion_model, measurement_model=sensor,
                                          obstacles=obstacles, _map=grid_map, sigma_v=1.0)
        bmdp.set_known_map(obstacles)
        vi = ValueIteration(bmdp, epsilon=epsilon)
        if not vi.load_results():
            vi.run()
            vi.save_results()
        V = np.asnumpy(vi.V) if hasattr(vi.V, 'get') else np.array(vi.V)
        v0 = float(V.flat[initial_belief_index]) if initial_belief_index < V.size else float(np.mean(V))
        results.append({'n': n, 'M': M, 'm_n': int(bmdp.SQ.m_n), 'cardinality': int(bmdp.BQ.cardinality),
                        'value_initial': v0, 'value_mean': float(np.mean(V)), 'iterations': int(vi.iteration_count)})
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved (n={n}, M={M}); {len(results)} total.")
    except Exception as e:
        print(f"Error (n={n}, M={M}): {e}")
        raise

In [ ]:
# 2D plot: Optimal cost (value) vs number of grid points (m_n), one curve per M
# Style similar to "Fig. 1. Optimal costs of the finite models when the initial state is x = 1."
valid = [r for r in results if r.get('value_initial') is not None]
if not valid:
    print("No valid results to plot.")
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    for M in M_list:
        pts = [(r['m_n'], r['value_initial']) for r in valid if r['M'] == M]
        pts.sort(key=lambda x: x[0])
        if pts:
            xs, ys = zip(*pts)
            ax.plot(xs, ys, 'D-', label=f'M = {M}', markersize=8)
    ax.set_xlabel('Number of grid points (m_n)')
    ax.set_ylabel('Optimal cost (value at initial belief)')
    ax.set_title('Optimal costs of the finite localization models (initial belief index 0)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / 'value_vs_grid_points_LOC.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 2D plot: Value vs state quantization n (one curve per M)
valid = [r for r in results if r.get('value_initial') is not None]
if not valid:
    print("No valid results to plot.")
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    for M in M_list:
        pts = [(r['n'], r['value_initial']) for r in valid if r['M'] == M]
        pts.sort(key=lambda x: x[0])
        if pts:
            xs, ys = zip(*pts)
            ax.plot(xs, ys, 'D-', label=f'M = {M}', markersize=8)
    ax.set_xlabel('State quantization (n)')
    ax.set_ylabel('Optimal cost (value at initial belief)')
    ax.set_title('Optimal costs vs state quantization n')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / 'value_vs_n_LOC.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 3D plot: (n, M, value)
valid = [r for r in results if r.get('value_initial') is not None]
if not valid:
    print("No valid results for 3D plot.")
else:
    from mpl_toolkits.mplot3d import Axes3D

    nn = [r['n'] for r in valid]
    MM = [r['M'] for r in valid]
    VV = [r['value_initial'] for r in valid]

    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(nn, MM, VV, s=80, c=VV, cmap='viridis')

    # Optional: surface if we have a grid of (n, M)
    n_arr = np.array(n_list)
    m_arr = np.array(M_list)
    if len(n_list) >= 2 and len(M_list) >= 2:
        V_grid = np.full((len(n_list), len(M_list)), np.nan)
        for r in valid:
            i = n_list.index(r['n'])
            j = M_list.index(r['M'])
            V_grid[i, j] = r['value_initial']
        if not np.all(np.isnan(V_grid)):
            nn_grid, MM_grid = np.meshgrid(n_arr, m_arr, indexing='ij')
            ax.plot_surface(nn_grid, MM_grid, V_grid, alpha=0.4, cmap='viridis')

    ax.set_xlabel('State quantization (n)')
    ax.set_ylabel('Belief quantization (M)')
    ax.set_zlabel('Optimal cost (value)')
    ax.set_title('Value function vs (n, M)')
    plt.tight_layout()
    fig.savefig(out_dir / 'value_3d_n_M_LOC.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Summary table
valid = [r for r in results if r.get('value_initial') is not None]
if valid:
    print(f"{'n':>3} {'M':>3} {'m_n':>8} {'card':>10} {'V(0)':>10} {'V_mean':>10} {'iters':>6}")
    print('-' * 55)
    for r in sorted(valid, key=lambda x: (x['n'], x['M'])):
        print(f"{r['n']:>3} {r['M']:>3} {r.get('m_n', 0):>8} {r.get('cardinality', 0):>10} "
              f"{r['value_initial']:>10.4f} {r.get('value_mean', 0):>10.4f} {r.get('iterations', 0):>6}")
else:
    print("No valid results.")